# Create an Agentic RAG using OpenVINO and LlamaIndex

An **agent** is an automated reasoning and decision engine. It takes in a user input/query and can make internal decisions for executing that query in order to return the correct result. The key agent components can include, but are not limited to:

- Breaking down a complex question into smaller ones
- Choosing an external Tool to use + coming up with parameters for calling the Tool
- Planning out a set of tasks
- Storing previously completed tasks in a memory module

[LlamaIndex](https://docs.llamaindex.ai/en/stable/) is a framework for building context-augmented generative AI applications with LLMs.LlamaIndex imposes no restriction on how you use LLMs. You can use LLMs as auto-complete, chatbots, semi-autonomous agents, and more. It just makes using them easier. You can build agents on top of your existing LlamaIndex RAG pipeline to empower it with automated decision capabilities. A lot of modules (routing, query transformations, and more) are already agentic in nature in that they use LLMs for decision making.

**Agentic RAG = Agent-based RAG implementation**

While standard RAG excels at simple queries across a few documents, agentic RAG takes it a step further and emerges as a potent solution for question answering. It introduces a layer of intelligence by employing AI agents. These agents act as autonomous decision-makers, analyzing initial findings and strategically selecting the most effective tools for further data retrieval. This multi-step reasoning capability empowers agentic RAG to tackle intricate research tasks, like summarizing, comparing information across multiple documents and even formulating follow-up questions -all in an orchestrated and efficient manner.

![agentic-rag](https://github.com/openvinotoolkit/openvino_notebooks/assets/91237924/871cb90d-27fd-4a87-aa3c-f4cdb199a148)

This example will demonstrate using RAG engines as a tool in an agent with OpenVINO and LlamaIndex.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Download models](#Download-models)
  - [Download LLM](#Download-LLM)
  - [Download Embedding model](#Download-Embedding-model)
- [Create models](#Create-models)
  - [Create OpenVINO LLM](#Create-OpenVINO-LLM)
  - [Create OpenVINO Embedding](#Create-OpenVINO-Embedding)
- [Create tools](#Create-tools)
- [Run Agentic RAG](#Run-Agentic-RAG)

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/llm-agent-rag-llamaindex/llm-agent-rag-llamaindex.ipynb" />


## Prerequisites

[back to top ⬆️](#Table-of-contents:)

Install required dependencies

In [1]:
# to avoid conflicts between versions
%pip uninstall -q -y optimum optimum-intel optimum-onnx

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import requests
from pathlib import Path

utility_files = ["notebook_utils.py", "cmd_helper.py", "pip_helper.py"]

for utility in utility_files:
    local_path = Path(utility)
    if not local_path.exists():
        r = requests.get(
            url=f"https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/{local_path.name}",
        )
        with local_path.open("w") as f:
            f.write(r.text)

os.environ["GIT_CLONE_PROTECTION_ACTIVE"] = "false"

from pip_helper import pip_install

pip_install(
    "-q",
    "--extra-index-url",
    "https://download.pytorch.org/whl/cpu",
    "git+https://github.com/huggingface/optimum-intel.git",
    "datasets<4.0.0",
    "accelerate",
    "nncf>=3",
    "llama-index",
    "llama-index-readers-file",
    "llama-index-core",
    "torch",
    "torchvision",
    "llama-index-llms-huggingface>=0.4.0,<0.5.0",
    "llama-index-embeddings-huggingface>=0.4.0,<0.5.0",
    "huggingface-hub>=0.26.5,<1.0",
)
pip_install("openvino>=2025.3.0", "openvino-tokenizers[transformers]")
pip_install("-q", "--no-deps", "llama-index-llms-openvino", "llama-index-embeddings-openvino")

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("llm-agent-rag-llamaindex.ipynb")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.29.1 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


  Using cached sentencepiece-0.2.2-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (33 kB)
Using cached sentencepiece-0.2.2-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (1.4 MB)


In [3]:
from notebook_utils import download_file

text_example_en_path = Path("text_example_en.pdf")
text_example_en = "https://github.com/user-attachments/files/16171326/xeon6-e-cores-network-and-edge-brief.pdf"


download_file(text_example_en, text_example_en_path)

PosixPath('/home/amd/Desktop/OpenVINO-Notebooks-on-AMD/results/llm-agent-rag-llamaindex/workdir-cpu/text_example_en.pdf')

## Download models

[back to top ⬆️](#Table-of-contents:)

### Download LLM

[back to top ⬆️](#Table-of-contents:)

To run LLM locally, we have to download the model in the first step. It is possible to [export your model](https://github.com/huggingface/optimum-intel?tab=readme-ov-file#export) to the OpenVINO IR format with the CLI, and load the model from local folder.

Large Language Models (LLMs) are a core component of agent. LlamaIndex does not serve its own LLMs, but rather provides a standard interface for interacting with many different LLMs. In this example, we can select `Phi-4-mini-instruct`, `Phi3-mini-instruct` or `Meta-Llama-3-8B-Instruct` as LLM in agent pipeline.
* **phi-4-mini-instruct** - Phi-4-mini-instruct is a 3.8B parameters, lightweight, state-of-the-art open model built upon synthetic data and filtered publicly available websites, with a focus on high-quality, reasoning dense data. It offers stronger instruction following and tool-use behavior than Phi-3-mini, which makes it a better default for reliable ReAct-style agents. More details about model can be found in [model card](https://huggingface.co/microsoft/Phi-4-mini-instruct).
* **phi3-mini-instruct** - The Phi-3-Mini is a 3.8B parameters, lightweight, state-of-the-art open model trained with the Phi-3 datasets that includes both synthetic data and the filtered publicly available websites data with a focus on high-quality and reasoning dense properties. More details about model can be found in [model card](https://huggingface.co/microsoft/Phi-3-mini-4k-instruct), [Microsoft blog](https://aka.ms/phi3blog-april) and [technical report](https://aka.ms/phi3-tech-report).
* **llama-3.1-8b-instruct** - The Llama 3.1 instruction tuned text only models (8B, 70B, 405B) are optimized for multilingual dialogue use cases and outperform many of the available open source and closed chat models on common industry benchmarks. More details about model can be found in [Meta blog post](https://ai.meta.com/blog/meta-llama-3-1/), [model website](https://llama.meta.com) and [model card](https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct).
>**Note**: run model with demo, you will need to accept license agreement. 
>You must be a registered user in 🤗 Hugging Face Hub. Please visit [HuggingFace model card](https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct), carefully read terms of usage and click accept button.  You will need to use an access token for the code below to run. For more information on access tokens, refer to [this section of the documentation](https://huggingface.co/docs/hub/security-tokens).
>You can login on Hugging Face Hub in notebook environment, using following code:
 
```python
    ## login to huggingfacehub to get access to pretrained model 

    from huggingface_hub import notebook_login, whoami

    try:
        whoami()
        print('Authorization token already provided')
    except OSError:
        notebook_login()
```

In [4]:
import ipywidgets as widgets

llm_model_ids = ["OpenVINO/Phi-4-mini-instruct-int4-ov", "OpenVINO/Phi-3-mini-4k-instruct-int4-ov", "meta-llama/Meta-Llama-3.1-8B-Instruct"]

llm_model_id = widgets.Dropdown(
    options=llm_model_ids,
    value=llm_model_ids[0],
    description="Model:",
    disabled=False,
)

llm_model_id

Dropdown(description='Model:', options=('OpenVINO/Phi-4-mini-instruct-int4-ov', 'OpenVINO/Phi-3-mini-4k-instru…

In [5]:
from pathlib import Path
import huggingface_hub as hf_hub
from cmd_helper import optimum_cli

llm_model_path = llm_model_id.value.split("/")[-1]
repo_name = llm_model_id.value.split("/")[0]

if not Path(llm_model_path).exists():
    if repo_name == "OpenVINO":
        hf_hub.snapshot_download(llm_model_id.value, local_dir=llm_model_path)
    else:
        optimum_cli(llm_model_id.value, llm_model_path, additional_args={"task": "text-generation-with-past", "weight-format": "int4"})

### Download Embedding model

[back to top ⬆️](#Table-of-contents:)

Embedding model is another key component in RAG pipeline. It takes text as input, and return a long list of numbers used to capture the semantics of the text. An OpenVINO embedding model and tokenizer can be exported by `feature-extraction` task with `optimum-cli`. In this tutorial, we use [bge-small-en-v1.5](https://huggingface.co/BAAI/bge-small-en-v1.5) as example.

In [6]:
embedding_model_id = "BAAI/bge-small-en-v1.5"
embedding_model_path = "bge-small-en-v1.5"

if not Path(embedding_model_path).exists():
    optimum_cli(embedding_model_id, embedding_model_path, additional_args={"task": "feature-extraction"})

## Create models

[back to top ⬆️](#Table-of-contents:)

### Create OpenVINO LLM

[back to top ⬆️](#Table-of-contents:)

Select device for LLM model inference

In [7]:
from notebook_utils import device_widget

llm_device = device_widget("CPU", exclude=["NPU"])

llm_device

Dropdown(description='Device:', options=('CPU', 'GPU', 'AUTO'), value='CPU')

OpenVINO models can be run locally through the `OpenVINOLLM` class in [LlamaIndex](https://docs.llamaindex.ai/en/stable/examples/llm/openvino/). If you have an Intel GPU, you can specify `device_map="gpu"` to run inference on it.

In [8]:
from llama_index.llms.openvino import OpenVINOLLM
from transformers import AutoTokenizer

import openvino.properties as props
import openvino.properties.hint as hints
import openvino.properties.streams as streams

ov_config = {hints.performance_mode(): hints.PerformanceMode.LATENCY, streams.num(): "1", props.cache_dir(): ""}


def phi_completion_to_prompt(completion):
    return f"<|system|><|end|><|user|>{completion}<|end|><|assistant|>\n"


def llama3_completion_to_prompt(completion):
    return f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{completion}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"


llm = OpenVINOLLM(
    model_id_or_path=str(llm_model_path),
    context_window=3900,
    max_new_tokens=1000,
    model_kwargs={"ov_config": ov_config},
    generate_kwargs={"do_sample": False, "temperature": None, "top_p": None},
    completion_to_prompt=phi_completion_to_prompt if "Phi" in llm_model_path else llama3_completion_to_prompt,
    device_map=llm_device.value,
)

# ReAct requires the model to stop right before "Observation:" so the framework can inject the real
# tool output. Otherwise small local models hallucinate their own observation and the agent loops
# until it hits max_iterations.
llm.generate_kwargs["stop_strings"] = ["Observation:"]
llm.generate_kwargs["tokenizer"] = AutoTokenizer.from_pretrained(str(llm_model_path))

/home/amd/Desktop/OpenVINO-Notebooks-on-AMD/.venvs/cpu/14c79f40de9d556c/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2274: UnsupportedFieldAttributeWarning: The 'validate_default' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'validate_default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(


  EXECUTION_DEVICES: ['CPU']


  KEY_CACHE_PRECISION: <Type: 'uint8_t'>


  PERF_COUNT: NO


  PERFORMANCE_HINT: LATENCY


...

  VALUE_CACHE_GROUP_SIZE: 0


  ENABLE_HYPER_THREADING: False


EXECUTION_DEVICES:


  CPU: AMD RYZEN AI MAX+ PRO 395 w/ Radeon 8060S      


The model `Phi-4-mini-instruct-int4-ov/` and tokenizer `Phi-4-mini-instruct-int4-ov` are different, please ensure that they are compatible.


### Create OpenVINO Embedding

[back to top ⬆️](#Table-of-contents:)

Select device for embedding model inference

In [9]:
embedding_device = device_widget()

embedding_device

Dropdown(description='Device:', options=('CPU', 'GPU', 'AUTO'), value='CPU')

A Hugging Face embedding model can be supported by OpenVINO through [`OpenVINOEmbeddings`](https://docs.llamaindex.ai/en/stable/examples/embeddings/openvino/) class of LlamaIndex.

In [10]:
from llama_index.embeddings.huggingface_openvino import OpenVINOEmbedding

embedding = OpenVINOEmbedding(model_id_or_path=embedding_model_path, device=embedding_device.value)

  EXECUTION_DEVICES: ['CPU']


  KEY_CACHE_PRECISION: <Type: 'uint8_t'>


  PERF_COUNT: NO


  PERFORMANCE_HINT: LATENCY


  SCHEDULING_CORE_TYPE: SchedulingCoreType.ANY_CORE


...

  ENABLE_CPU_RESERVATION: False


  VALUE_CACHE_GROUP_SIZE: 0


  ENABLE_HYPER_THREADING: False


EXECUTION_DEVICES:


  CPU: AMD RYZEN AI MAX+ PRO 395 w/ Radeon 8060S      


## Create tools

[back to top ⬆️](#Table-of-contents:)

In this examples, we will create 2 customized tools for `multiply` and `add`.

In [11]:
from llama_index.core.agent.workflow import ReActAgent
from llama_index.core.tools import FunctionTool


def multiply(a: float, b: float) -> float:
    """Multiply two numbers and returns the product"""
    return a * b


multiply_tool = FunctionTool.from_defaults(fn=multiply)


def divide(a: float, b: float) -> float:
    """Add two numbers and returns the sum"""
    return a / b


divide_tool = FunctionTool.from_defaults(fn=divide)

To demonstrate using RAG engines as a tool in an agent, we're going to create a very simple RAG query engine as one of the tools. 

>**Note**: For a full RAG pipeline with OpenVINO, you can check the [RAG notebooks](../llm-rag-llamaindex)

In [12]:
from llama_index.core import SimpleDirectoryReader
from llama_index.core import VectorStoreIndex, Settings

Settings.embed_model = embedding
Settings.llm = llm

reader = SimpleDirectoryReader(input_files=[text_example_en_path])
documents = reader.load_data()
index = VectorStoreIndex.from_documents(
    documents,
)

Now we turn our query engine into a tool by supplying the appropriate metadata (for the python functions, this was being automatically extracted so we didn't need to add it):

In [13]:
from llama_index.core.tools import QueryEngineTool, ToolMetadata

vector_tool = QueryEngineTool(
    index.as_query_engine(streaming=True),
    metadata=ToolMetadata(
        name="vector_search",
        description="Useful for searching for basic facts about 'Intel Xeon 6 processors'",
    ),
)

## Run Agentic RAG

[back to top ⬆️](#Table-of-contents:)

We modify our agent by adding this engine to our array of tools (we also remove the llm parameter, since it's now provided by settings):

In [14]:
agent = ReActAgent(tools=[multiply_tool, divide_tool, vector_tool], llm=llm)

Ask a question using multiple tools.

In [15]:
from llama_index.core.agent.workflow import AgentStream, ToolCallResult

handler = agent.run("What's the maximum number of cores of 8 sockets of 'Intel Xeon 6 processors' ? Go step by step, using a tool to do any math.")

async for ev in handler.stream_events():  # noqa: F704
    if isinstance(ev, ToolCallResult):
        print(f"\nCalled tool '{ev.tool_name}' -> {ev.tool_output}\n")
    elif isinstance(ev, AgentStream):
        print(ev.delta, end="", flush=True)

response = await handler  # noqa: F704
print(str(response))

Thought: 

To 

find 

the 

maximum 

...

6 

processors' 

is 

1152.

The maximum number of cores in 8 sockets of 'Intel Xeon 6 processors' is 1152.
